In [1]:
import numpy as np
import pandas as pd
from numpy import nan
from sklearn.preprocessing import  StandardScaler, OneHotEncoder, MinMaxScaler
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.datasets import make_classification 
from sklearn.metrics import classification_report, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA, TruncatedSVD
from nltk.stem import SnowballStemmer
from sklearn import linear_model, datasets
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from sklearn.svm import LinearSVC, LinearSVR
from sklearn.feature_selection import SelectKBest, SelectFpr, chi2, mutual_info_classif, f_classif
from sklearn.linear_model import SGDClassifier
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem import SnowballStemmer
from sklearn.naive_bayes import MultinomialNB
from sklearn.calibration import CalibratedClassifierCV
import re
import string
from bs4 import BeautifulSoup
from sklearn.svm import LinearSVC, SVC
from sklearn.metrics import confusion_matrix
import numpy as np
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.base import BaseEstimator, TransformerMixin

In [2]:
df = pd.read_csv("winter_project_2026/development.csv")
eva = pd.read_csv("winter_project_2026/evaluation.csv")

In [3]:
mask = ((df['title'].isin(eva['title'])) & (df['article'].isin(eva['article'])) & (df['source'].isin(eva['source'])))
df[mask].groupby('label').count()
#df.drop(df[mask].index, inplace=True) #remove data leakage!!!!!!!?!?!?!??!?!!!

,Id,source,title,article,page_rank,timestamp
label,,,,,,
0,328,328,328,328,328,328
1,163,163,163,163,163,163
2,134,134,134,134,134,134
3,306,306,306,306,306,306
4,53,53,53,53,53,53
5,504,504,504,504,504,504
6,93,93,93,93,93,93


In [4]:
df.drop_duplicates(inplace=True)
df = df.dropna()
test_ids = eva['Id'].copy()
eva['article'].fillna("", inplace=True)

/var/folders/j9/m1q0h7955ls67lg17_lw___80000gn/T/ipykernel_67437/2900358585.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  eva['article'].fillna("", inplace=True)


In [5]:
import re
import string
from bs4 import BeautifulSoup, Comment
import re

def clean_text(text):
    text = text.lower()  # Lowercase
    text = re.sub(r'<[^>]+>', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    text = re.sub(r'\W', ' ', text)  # Remove special characters
    text = BeautifulSoup(text, "html.parser").get_text()  # Remove HTML tags
    return text

def clean_html_professional(html_content):
    if not isinstance(html_content, str):
        return ""
    soup = BeautifulSoup(html_content, "html.parser")
    meta_desc = ""
    meta_tag = soup.find('meta', attrs={'name': 'description'})
    if meta_tag and 'content' in meta_tag.attrs:
        meta_desc = meta_tag['content']
    for tag in soup(['script', 'style', 'noscript', 'iframe', 'svg', 'header', 'footer', 'nav']):
        tag.decompose()
    comments = soup.find_all(string=lambda text: isinstance(text, Comment))
    for c in comments:
        c.extract()
    for img in soup.find_all('img'):
        if 'alt' in img.attrs and img['alt']:
            img.replace_with(f" {img['alt']} ") 
    text = soup.get_text(separator=' ', strip=True)
    final_text = f"{text} {meta_desc}"
    final_text = re.sub(r'\s+', ' ', final_text).strip()
    return final_text

for i, article in zip(df.index, df['article']):
    df.loc[i, 'article'] = clean_html_professional(article)

#for i, article in zip(eva.index, eva['article']):
#    eva.loc[i, 'article'] = clean_html_professional(article)


In [7]:
# unisco le colonne titolo e articolo in una colonna unica raddoppiando l'importanza del titolo
#eva['text'] = eva['source'] + ' ' + eva['source'] + ' ' + eva['title'] + ' ' + eva['title'] + ' ' + eva['title'] + ' ' + eva['article'] + ' ' + eva['article']
#df['text'] = df['source'] + ' ' + df['title'] + ' ' + df['article']
df['text'] = df['source'] + ' ' + df['source'] + ' ' + df['title'] + ' ' + df['title'] + ' ' + df['title'] + ' ' + df['article'] + ' ' + df['article']
df.columns

Index(['Id', 'source', 'title', 'article', 'page_rank', 'timestamp', 'label',
       'text'],
      dtype='object')

In [8]:
df['text_word_count'] = df['text'].apply(lambda x: len(str(x).split()))
df['title_word_count'] = df['title'].apply(lambda x: len(str(x).split()))

In [9]:
y = df['label']
df.drop('label', inplace=True, axis=1)

In [10]:
X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.2, random_state=42, stratify=y, shuffle=True)

In [28]:
stop_words = list(ENGLISH_STOP_WORDS) + [
    # --- Rumore HTML e Scraping ---
    'http', 'https', 'src', 'href', 'img', 'alignleft', 'alignright', 
    'height', 'width', 'border', 'clearall', 'borderimga', 'br', 'class', 'Reuters']

encoder = OneHotEncoder(min_frequency=70, handle_unknown='ignore')
standardizer = StandardScaler()

vectorizer_svd = Pipeline([( 'vect', TfidfVectorizer(
        max_features=30000,
        stop_words='english',
        ngram_range=(1, 2),
        min_df=3)),
        ('svd', TruncatedSVD(
            n_components=1000, 
            algorithm='randomized'))
        ])

vectorizer = TfidfVectorizer(
        max_features=60000,
        stop_words='english',
        ngram_range=(1, 2),
        min_df=3)

vectorizer2 = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3,5),
    min_df=5,
    max_features=30000
)

preprocessor = ColumnTransformer(transformers=[
    ('text_char',vectorizer2, 'text'),
    ('text_svd',vectorizer, 'text'),
], remainder='drop', n_jobs=-1)
    #('numerical', standardizer, ['text_word_count', 'repetition_rate', 'day_of_week', 'moment_of_day'])
    #('text',vectorizer, 'text'),


In [29]:
X_train_final = preprocessor.fit_transform(X_train)
X_test_final = preprocessor.transform(X_test)

In [30]:
import numpy as np
from sklearn.base import clone
from sklearn.metrics import classification_report, f1_score

# --- STEP 1: PREPARAZIONE DATI ---

# 1. Creiamo le label per il SUPER-MODELLO (Merge 0 e 5)
# Trasformiamo tutte le label 5 in 0. 
# Ora la classe 0 rappresenta il concetto "News Generiche/Internazionali"
y_train_super = y_train.copy()
y_train_super[y_train == 5] = 0 

# 2. Creiamo i dati per il SUB-MODELLO (Solo 0 vs 5)
# Prendiamo solo le righe che sono ORIGINARIAMENTE 0 o 5
df['label']=y
mask_sub = df['label'].isin([0, 5])  # Usa df['label'] o y_train se è una Series
# Attenzione: assicurati di filtrare X_train_final corrispondente
# Se X_train_final è una matrice sparsa/array, usiamo la maschera booleana su y_train per filtrare
mask_sub_train = np.isin(y_train, [0, 5])

X_train_sub = X_train_final[mask_sub_train]
y_train_sub = y_train[mask_sub_train]

print(f"Super-Model Labels: {np.unique(y_train_super)}") # Dovrebbe mancare il 5
print(f"Sub-Model shape: {X_train_sub.shape}") # Dataset più piccolo

Super-Model Labels: [0 1 2 3 4 6]
Sub-Model shape: (29275, 90000)


In [31]:
# --- STEP 2: TRAINING ---

svm_clf = SGDClassifier(
    loss='log_loss', 
    penalty='l2',
    alpha=1e-5,
    max_iter=2000,
    tol=1e-6,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)

# Modello Padre (Super-Classifier)
# Deve distinguere tra [0+5], 1, 2, 3, 4, 6
super_clf = clone(svm_clf)
print("Addestramento Super-Model (Macro Categorie)...")
super_clf.fit(X_train_final, y_train_super)

sub_clf = clone(svm_clf)
# Consiglio: Per il binario 0vs5, magari forza pesi bilanciati
# sub_clf.estimators[1][1].set_params(class_weight='balanced') 
print("Addestramento Sub-Model (Specialista 0 vs 5)...")
sub_clf.fit(X_train_sub, y_train_sub)

Addestramento Super-Model (Macro Categorie)...
Addestramento Sub-Model (Specialista 0 vs 5)...


,loss,'log_loss'
,penalty,'l2'
,alpha,1e-05
,l1_ratio,0.15
,fit_intercept,True
,max_iter,2000
,tol,1e-06
,shuffle,True
,verbose,0
,epsilon,0.1
,n_jobs,-1


In [32]:
import numpy as np
from sklearn.metrics import classification_report, f1_score

mask_test_sub = np.isin(y_test, [0, 5])
X_test_sub_eval = X_test_final[mask_test_sub]
y_test_sub_eval = y_test[mask_test_sub]

print("--- SUB-MODEL EVALUATION (0 vs 5 ONLY) ---")
y_pred_sub_only = sub_clf.predict(X_test_sub_eval)
print(classification_report(y_test_sub_eval, y_pred_sub_only))

def hierarchical_predict(X):
    y_pred_super = super_clf.predict(X)
    
    mask_ambiguous = (y_pred_super == 0)
    
    if not np.any(mask_ambiguous):
        return y_pred_super
    
    X_ambiguous = X[mask_ambiguous]
    y_pred_sub = sub_clf.predict(X_ambiguous)
    
    y_final = y_pred_super.copy()
    y_final[mask_ambiguous] = y_pred_sub
    
    return y_final

print("--- FINAL HIERARCHICAL EVALUATION ---")
y_pred_hierarchical = hierarchical_predict(X_test_final)

print(classification_report(y_test, y_pred_hierarchical))
print(f"Macro F1: {f1_score(y_test, y_pred_hierarchical, average='macro'):.4f}")
print(confusion_matrix(y_test, y_pred_hierarchical))


--- SUB-MODEL EVALUATION (0 vs 5 ONLY) ---
              precision    recall  f1-score   support

           0       0.84      0.77      0.80      4709
           5       0.64      0.74      0.68      2611

    accuracy                           0.76      7320
   macro avg       0.74      0.75      0.74      7320
weighted avg       0.77      0.76      0.76      7320

--- FINAL HIERARCHICAL EVALUATION ---
              precision    recall  f1-score   support

           0       0.75      0.68      0.71      4709
           1       0.73      0.81      0.77      2117
           2       0.80      0.81      0.80      2232
           3       0.61      0.50      0.55      1995
           4       0.78      0.93      0.85      1715
           5       0.50      0.51      0.51      2611
           6       0.61      0.76      0.68       620

    accuracy                           0.70     15999
   macro avg       0.68      0.72      0.70     15999
weighted avg       0.69      0.70      0.69     15

In [33]:
maximum = 5
# Trova gli indici degli errori specifici
mask_error = (y_test == 5) & (y_pred_hierarchical == 5)
X_error_texts = df.loc[y_test.index[mask_error], 'article'] # O il nome della colonna testo

print("--- TESTI general PREDETTI COME general ---")
for text in X_error_texts.head(maximum):
    print(text[:300])
    print("----------------")
    
    
mask_error = (y_test == 0) & (y_pred_hierarchical == 0)
X_error_texts = df.loc[y_test.index[mask_error], 'article'] # O il nome della colonna testo

print("--- TESTI inter PREDETTI COME inter ---")
for text in X_error_texts.head(maximum):
    print(text[:300])
    print("----------------")
    

mask_error = (y_test == 0) & (y_pred_hierarchical == 5)
X_error_texts = df.loc[y_test.index[mask_error], 'article'] # O il nome della colonna testo

print("--- TESTI inter PREDETTI COME general ---")
for text in X_error_texts.head(maximum):
    print(text[:300])
    print("----------------")
    
    
mask_error = (y_test == 5) & (y_pred_hierarchical == 0)
X_error_texts = df.loc[y_test.index[mask_error], 'article'] # O il nome della colonna testo

print("--- TESTI general PREDETTI COME inter ---")
for text in X_error_texts.head(maximum):
    print(text[:300])
    print("----------------")

--- TESTI general PREDETTI COME general ---
BAGHDAD (Reuters) - Violence in Iraq has fallen to its lowest level since before a 2006 mosque attack which unleashed the deadliest phase of the Iraq war, the deputy commander of U.S. forces in Iraq said on Thursday.
----------------
On CNN
----------------
Ukraine's Supreme Court took a step closer to an anxiously awaited ruling on the validity of the disputed presidential runoff, but adjourned for the day Thursday without a decision.
----------------
$150,000 mortgage for only $610/month. Lower your mortgage - free quotes.
----------------
Arts & entertainment: Joel and Ethan Coen tell John Patterson about Texas, torture and a 'fantastic' haircut
----------------
--- TESTI inter PREDETTI COME inter ---
Bombs struck workers in a predominantly Shiite area on the southeastern edge of Baghdad, killing at least eight people and wounding two dozen.
----------------
The EU sought Wednesday to keep pressure on Turkey over its bid to start talks on 